In [6]:
from dataclasses import dataclass	
import polars as pl
import os
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
if 'notebooks' in str(PROJECT_DIR):
	PROJECT_DIR = PROJECT_DIR.parent

if PROJECT_DIR not in sys.path:
	sys.path.insert(1, str(PROJECT_DIR))

In [7]:
from src_strategy.configs.dataconfig import input_output_config_3_1

In [10]:
os.listdir(input_output_config_3_1.output_path)

['df_sirs.parquet',
 'df_suspected_infection.parquet',
 'meta',
 'df_aggregated_map_measured_only.parquet',
 'backbone.parquet',
 'df_all.parquet',
 'df_aggregated.parquet',
 'df_encounters.parquet',
 'df_all_map_measured_only.parquet',
 'df_all_no_collisions.parquet',
 'df_all_no_collisions_map_measured_only.parquet']

In [11]:
df_agg = pl.read_parquet(
    os.path.join(
        input_output_config_3_1.output_path,
		'df_aggregated_map_measured_only.parquet'
	)
)

In [12]:
df_agg

EncounterEpicCsn,Event_DateTime,last_temp_8h,last_temp_8h_source_ts,last_pulse_8h,last_pulse_8h_source_ts,last_resp_8h,last_resp_8h_source_ts,last_sbp_8h,last_sbp_8h_source_ts,last_map_8h,last_map_8h_source_ts,last_wbc_12h,last_wbc_12h_source_ts,last_creatinine_12h,last_creatinine_12h_source_ts,last_egfr_12h,last_egfr_12h_source_ts,last_bilirubin_12h,last_bilirubin_12h_source_ts,last_lactate_6h,last_lactate_6h_source_ts,last_platelets_24h,last_platelets_24h_source_ts,last_inr_12h,last_inr_12h_source_ts,last_aptt_24h,last_aptt_24h_source_ts,last_gcs_12h,last_gcs_12h_source_ts,last_vasopressin_24h,last_vasopressin_24h_last_set_ts,last_phenylephrine_24h,last_phenylephrine_24h_last_set_ts,last_norepinephrine_24h,last_norepinephrine_24h_last_set_ts,last_epinephrine_24h,last_epinephrine_24h_last_set_ts,last_pfratio_2h,last_pfratio_2h_source_ts
i64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],f64,datetime[μs],i32,datetime[μs],i32,datetime[μs],i32,datetime[μs],i32,datetime[μs],f64,datetime[μs]
659308243,2022-06-14 10:50:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,0,null,0,null,0,null,null,null
659308243,2022-06-14 11:22:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,0,null,0,null,0,null,null,null
659308243,2022-06-14 11:23:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,0,null,0,null,0,null,null,null
659308243,2022-06-14 11:52:00,98.2,2022-06-14 11:52:00,76.0,2022-06-14 11:52:00,18.0,2022-06-14 11:52:00,138.0,2022-06-14 11:52:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,0,null,0,null,0,null,null,null
659308243,2022-06-14 13:21:00,98.2,2022-06-14 11:52:00,76.0,2022-06-14 11:52:00,18.0,2022-06-14 11:52:00,138.0,2022-06-14 11:52:00,null,null,null,null,null,null,null,null,null,null,1.6,2022-06-14 13:21:00,null,null,null,null,null,null,null,null,0,null,0,null,0,null,0,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
753774459,2026-05-28 15:00:00,null,null,null,null,null,null,132.0,2026-05-28 15:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,0,null,0,null,0,null,null,null
753774459,2026-05-28 19:27:00,null,null,null,null,null,null,149.0,2026-05-28 19:27:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,0,null,0,null,0,null,null,null
753774459,2026-05-28 22:28:00,null,null,null,null,null,null,141.0,2026-05-28 22:28:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,0,null,0,null,0,null,null,null


## Design Configuration for SIRS
SIRS consists of 4 components: **Temperature**, **WBC**, **Respiration Rate**, and **Pulse** <br>

### Bounds
- Temp > 100.4 OR Temp < 96.8
- HR > 90
- RR > 20
- WBC > 12 OR WBC < 4

### Criteria
- Each bound violation is counted as 1
- Total number of 2 or more violation is considered Positive SIRS


In [22]:
[c for c in df_agg.columns if 'resp' in c.lower()]

['last_resp_8h', 'last_resp_8h_source_ts']

In [23]:
from dataclasses import dataclass

@dataclass
class SIRSConfig:
	temp_col_name: str = "last_temp_8h"
	wbc_col_name: str = "last_wbc_8h"
	rr_col_name: str = "last_resp_8h"
	hr_col_name: str = "last_pulse_8h"

	temp_lower_bound: float = 96.8
	temp_upper_bound: float = 100.4
	temp_alias: str = "temp_flag"

	wbc_lower_bound: float = 4
	wbc_upper_bound: float = 12
	wbc_alias: str = "wbc_flag"

	hr_upper_bound: float = 90
	hr_alias: str = "hr_flag"

	rr_upper_bound: float = 20
	rr_alias: str = "rr_flag"


	sirs_output_col: str = 'sirs_score'

In [26]:
from abc import ABC, abstractmethod
from typing import List
from typing import Protocol

class Reducer(Protocol):
	def reduce(self, df: pl.DataFrame, positive_cols: List[str], output_col_name: str):
		...


class SumReducer:
	def reduce(self, df: pl.DataFrame, positive_cols: List[str], output_col_name: str):
		for c in positive_cols:
			assert c in df.columns, f'Column {c} is missing'	
		return df.with_columns(
			pl.sum_horizontal(
				positive_cols
			).alias(output_col_name)
		)


class Criterion(ABC):
	def __init__(self, config, required_cols: List[str], enabled: bool):
		self.config = config
		self.required_cols = required_cols
		self.enabled(enabled)
	
	def _check_required_cols(self, df: pl.DataFrame | pl.LazyFrame):
		if isinstance(df, pl.DataFrame):
			in_cols = df.columns
		elif isinstance(df, pl.LazyFrame):
			in_cols = df.collect_schema().keys()
		else:
			raise TypeError("df is expected to be either pl.DataFrame or pl.LazyFrame")
		for c in self.required_cols:
			assert c in in_cols, f'Column {c} is a required column for WBCCriterion and it is missing'

	@property
	def enabled(self, enabled: bool = True):
		self.enabled = enabled
		return self.enabled

	@abstractmethod	
	def prepare_df(self, df: pl.DataFrame) -> pl.DataFrame:
		...

	@abstractmethod
	def expression(self, df: pl.DataFrame) -> pl.Expr:
		...
	

class Pipeline:
	def __init__(self, criteria: List[Criterion]):
		self.criteria = criteria

	def run(self, df: pl.DataFrame) -> pl.DataFrame:
		expr_list = []
		for criterion in self.criteria:
			if not criterion.enabled: continue
			df = criterion.prepare_df(df)
			expr = criterion.expression(df)
			expr_list.append(expr)
		df = df.with_columns(
			expr_list
		)
		return df


class TemperatureCriterion(Criterion):
	def __init__(self, config, required_cols, enabled):
		super().__init__(config, required_cols, enabled)

	def prepare_df(self, df:pl.DataFrame) -> pl.DataFrame:
		return df

	def expression(self)  -> pl.Expr:
		expr = (
			(pl.col(self.config.temp_col_name)<self.config.temp_lower_bound)
			|
			(pl.col(self.config.temp_col_name)>self.config.temp_upper_bound)
		).alias(self.config.temp_flag)

		return expr


class PulseCriterion(Criterion):
	def __init__(self, config, required_cols, enabled):
		super().__init__(config, required_cols, enabled)

	def prepare_df(self, df:pl.DataFrame) -> pl.DataFrame:
		return df

	def expression(self) -> pl.Expr:
		expr = (
			(pl.col(self.config.hr_col_name)>self.config.hr_upper_bound)
		).alias(self.config.hr_flag)

		return expr


class WBCCriterion(Criterion):
	def __init__(self, config, required_cols, enabled):
		super().__init__(config, required_cols, enabled)

	def prepare_df(self, df:pl.DataFrame) -> pl.DataFrame:
		self._check_required_cols(df)
		return df

	def expression(self) -> pl.Expr:
		expr = (
			(pl.col(self.config.wbc_col_name)<self.config.wbc_lower_bound)
			|
			(pl.col(self.config.wbc_col_name)>self.config.wbc_upper_bound)
		).alias(self.config.hr_flag)

		return expr


class RRCriterion(Criterion):
	def __init__(self, config, required_cols, enabled):
		super().__init__(config, required_cols, enabled)

	def prepare_df(self, df:pl.DataFrame) -> pl.DataFrame:
		self._check_required_cols(df)
		for col in self.required_cols:
			assert col in df.columns, f'Column {col} is a required column for RRCriterion and it is missing'
		return df

	def expression(self) -> pl.Expr:
		expr = (
			(pl.col(self.config.rr_col_name)>self.config.rr_upper_bound)
		).alias(self.config.rr_flag)

		return expr


sirs_config = SIRSConfig()
SIRS_CRITERIA_LIST = [
	TemperatureCriterion(sirs_config, required_cols=[sirs_config.temp_col_name, sirs_config.temp_alias], enabled=True),
	PulseCriterion(sirs_config, required_cols=[sirs_config.hr_col_name, sirs_config.hr_alias], enabled=True),
	WBCCriterion(sirs_config, required_cols=[sirs_config.wbc_col_name, sirs_config.wbc_alias], enabled=True),
	RRCriterion(sirs_config, required_cols=[sirs_config.rr_col_name, sirs_config.rr_alias], enabled=True),
]

def build_pipeline(criteria_list: List[Criterion]):
	active_criteria_list = [
		c for c in criteria_list if c.enabled
	]
	return Pipeline(
		active_criteria_list
	)
	



AttributeError: can't set attribute 'enabled'